$$
\newcommand{\dd}{\text{d}}
\newcommand{\pdv}[2]{ \frac{\partial #1}{\partial #2}  }
\newcommand{\dv}[2]{ \frac{\dd #1}{\dd #2} }
$$

# Softglass equation learning

Original work by __Jared Callaham (2020)__

Adapted to Softglass by __Zach McBrearty (2025)__

In [1]:
# Numpy
import numpy as np
from numpy.linalg import lstsq

# Matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl

# Sympy
import sympy

# Custom packages
import utils
import fpsolve

import data_loader as dl

## Load data from .npy

In [2]:
folder_path = dl.SCRATCH_PATH / "runs_EqnLearning/run_0D_0D_5_1_004"
step = 1
metadata, time, phi, sigma = dl.get_data(
    folder_path, start=100_000, stop=10_100_000, step=step
)
timeseries = np.concat([phi, sigma], axis=1)

dt = metadata["dt"] * step
R = metadata["R"]
ep0 = metadata["epsilon_0"]
ep1 = metadata["epsilon_1"]
lb = metadata["lb"]
m = lambda y: (y - 1) / np.sqrt(y) if y > 1 else 0.0
f = lambda x, y: -R * x + (m(y) - np.abs(x)) * x**3
g = lambda x, y: np.sqrt(ep0 + ep1 * x**2)

sigma_avg = np.average(sigma)
fluidity = phi**2

KeyboardInterrupt: 

In [ ]:
# Plot solution
fig, (ax, ax2) = plt.subplots(2)
ax: plt.Axes  # type: ignore
ax2: plt.Axes  # type: ignore
ax.plot(time, phi)
ax.set_xlabel("$t$")
ax.set_ylabel(r"Phi, $\phi$")

ax2.plot(time, sigma)
ax2.set_xlabel("$t$")
ax2.set_ylabel(r"Stress, $\sigma$")

fig.tight_layout()

fig.savefig(folder_path / "system_graph.png")

Log[fluidity] version

This doesn't fully work as $\mathbb{E}[log f]$ cannot be transformed to $\mathbb{E}[f]$.
$$
\mathbb{E}[\log(f)] \leqslant \log(\mathbb{E}[f])
$$
also
$$
    \mathbb{E}[\log(f)] = \int  \log f P[\log f] d[\log f] = \int \log f P[f] df \neq \int f P[f] df = \mathbb{E}[f]
$$

$P[\phi]$

In [ ]:
from kramersmoyal import km


def km_avg(data, tau, pdf_cutoff=np.e**3):
    kmc, centers = km(data, powers=2)  # type: ignore

    pdf = kmc[0]
    mask = pdf > pdf.max() / pdf_cutoff
    pdf = pdf[mask]
    pdf /= np.sum(pdf)

    centers = centers[0][mask]
    moment1, moment2 = kmc[1:, mask] / tau
    return centers, pdf, moment1, moment2


centers, pdf, moment1, moment2 = km_avg(phi, tau=dt, pdf_cutoff=np.e**5)
centers_skip500, pdf_skip500, moment1_skip500, moment2_skip500 = km_avg(
    phi[::500], tau=500 * dt, pdf_cutoff=np.e**5
)
fig, axes = plt.subplots(4, figsize=(12, 12))
axes: list[plt.Axes]  # type: ignore
axes[0].plot(time, phi)
axes[0].set_title("Process")

axes[1].plot(centers, pdf, label=rf"$\tau={dt}$")
axes[1].plot(centers_skip500, pdf_skip500, label=rf"$\tau={500*dt}$")
axes[1].set_title("PDF")
axes[1].legend()

axes[2].plot(centers, moment1, label=rf"$\tau={dt}$")
axes[2].plot(centers_skip500, moment1_skip500, label=rf"$\tau={500*dt}$")
axes[2].plot(centers, f(centers, sigma_avg), label="$A(f)$")
axes[2].set_title("First moment")
axes[2].legend()

axes[3].plot(centers, moment2, label=rf"$\tau={dt}$")
axes[3].plot(centers_skip500, moment2_skip500, label=rf"$\tau={500*dt}$")
axes[3].plot(centers, g(centers, sigma_avg) ** 2 / 2, label="$B(f)^2/2$")
axes[3].set_title("Second moment")
axes[3].legend()

fig.tight_layout()

fig.savefig(folder_path / "process_pdf_moments.png")

# Adjoint optimization

Correcting the finite-time distortion with the adjoint Fokker-Planck equation.  The routines to call the optimizer and the cost function are in `utils.py`.

In [ ]:
## Kramers-Moyal average
N = len(centers)
f_KM = moment1
a_KM = moment2

In [ ]:
### Build SINDy libraries with sympy
x = sympy.symbols("x")

f_expr = np.array(
    [x ** (2 * i + 1) for i in np.arange(0, 3)]
    + [sympy.Abs(x) * x ** (2 * i + 1) for i in np.arange(0, 3)]
)  # Polynomial library for drift
s_expr = np.array([x**i for i in np.arange(3)])  # Polynomial library for diffusion

# Convert sympy expressions into library matrices
lib_f = np.zeros([len(f_expr), N])
for k in range(len(f_expr)):
    lamb_expr = sympy.lambdify(x, f_expr[k])
    lib_f[k] = lamb_expr(centers)

lib_s = np.zeros([len(s_expr), N])
for k in range(len(s_expr)):
    lamb_expr = sympy.lambdify(x, s_expr[k])
    lib_s[k] = lamb_expr(centers)

In [ ]:
# Initialize Xi with least squares regression (no finite-time corrections)

Xi0 = np.zeros((len(f_expr) + len(s_expr)))
mask = np.nonzero(np.isfinite(f_KM))[0]
Xi0[: len(f_expr)] = lstsq(lib_f[:, mask].T, f_KM[mask], rcond=None)[
    0
]  # Regression against drift
Xi0[len(f_expr) :] = lstsq(lib_s[:, mask].T, np.sqrt(2 * a_KM[mask]), rcond=None)[
    0
]  # Regression against diffusion

print("Xi0 =", Xi0)

#### SSR for model selection

In [ ]:
# Initialize adjoint solver
afp = fpsolve.AdjFP(centers)

# Initialize forward steady-state solver
dx = centers[1] - centers[0]
fp = fpsolve.SteadyFP(N, dx)

# Optimization parameters
params = {
    "W": np.ones((2, N)),
    "f_KM": f_KM,
    "a_KM": a_KM,
    "Xi0": Xi0,
    "f_expr": f_expr,
    "s_expr": s_expr,
    "lib_f": lib_f.T,
    "lib_s": lib_s.T,
    "N": N,
    "kl_reg": 10,
    "fp": fp,
    "afp": afp,
    "p_hist": pdf,
    "tau": dt,
    "radial": False,
}

# Use anonymous function to automatically pass the cost function
opt_fun = lambda params: utils.AFP_opt(utils.cost, params)
Xi, V = utils.SSR_loop(opt_fun, params)

In [ ]:
####################
# SSR cost function
####################

labels = [r"${0}$".format(sympy.latex(t)) for t in np.concatenate((f_expr, s_expr))]

active = abs(Xi) > 1e-8

n_terms = len(labels)

fig, (ax, ax2) = plt.subplots(ncols=2, figsize=(12, 4))
ax: plt.Axes  # type: ignore
ax2: plt.Axes  # type: ignore
ax.scatter(np.arange(len(V)), V, c="k")
ax.set_xticks(np.arange(n_terms - 1))
ax.set_xticklabels(np.arange(n_terms, 1, -1))
ax.set_xlabel("Sparsity")
ax.set_ylabel(r"Cost")

ax2.pcolor(active, cmap="bone_r", edgecolors="gray")
ax2.set_yticks(0.5 + np.arange(n_terms))
ax2.set_yticklabels(labels)
ax2.set_xticks(0.5 + np.arange(n_terms - 1))
ax2.set_xticklabels(np.arange(n_terms, 1, -1))
ax2.set_xlabel("Sparsity")
ax2.set_ylabel("Active terms")

fig.savefig(folder_path / "SSR_sparsity.png")

In [ ]:
# Select model with the fewest terms before the cost function spikes
n_terms = 6
print(Xi[:, 1 - n_terms])
print(V[1 - n_terms])
Xi_f = Xi[: len(f_expr), 1 - n_terms]
Xi_s = Xi[len(f_expr) :, 1 - n_terms]

# Functions from the expressions
f_sym = utils.sindy_model(Xi_f, f_expr)
f_sindy = sympy.lambdify(x, f_sym)
s_sym = utils.sindy_model(Xi_s, s_expr)
a_sindy = sympy.lambdify(x, 0.5 * s_sym**2)

print(f"dphi = ({f_sym}) dt + ({s_sym}) dbeta")

f_vals = f_sindy(centers)
a_vals = a_sindy(centers)

# Check if a scalar (happens when library is a constant)
if np.ndim(f_vals) == 0:
    f_vals = f_vals + 0 * centers
if np.ndim(a_vals) == 0:
    a_vals = a_vals + 0 * centers

In [ ]:
# Compare PDFs: empirical vs Fokker-Planck solution with model

p_fit = fp.solve(f_vals, a_vals)
print(
    "KL divergence (LINDy model): {0:0.5f}".format(
        utils.kl_divergence(pdf, p_fit, dx=dx, tol=1e-6)
    )
)

fig, ax = plt.subplots(figsize=(5, 5))
ax: plt.Axes  # type: ignore
ax.plot(centers, pdf, "k", label="Data", lw=3)
ax.plot(centers, p_fit, "--", c="r", label="Model", lw=3)
ax.legend(fontsize=14)
ax.set_xlabel("$x$", fontsize=24)
ax.set_ylabel("$p(x)$", fontsize=24)

fig.savefig(folder_path / "pdf_comparision.png")

#### Predicted finite-time evolution of Kramers-Moyal coefficients

In particular, we see how the apparent state-dependent diffusion can arise from constant diffusion and coarse sampling

In [ ]:
afp.precompute_operator(f_vals, a_vals)
f_tau, a_tau = afp.solve(dt)

fig, (ax, ax2) = plt.subplots(ncols=2, figsize=(10, 5))
ax: plt.Axes  # type: ignore
ax2: plt.Axes  # type: ignore
ax.plot(centers, f(centers, sigma_avg), c="gray", lw=2, label="True drift")
ax.plot(centers, f_KM, ls="", marker=".", markersize=8, c="k")
ax.plot(centers, f_vals, "k", lw=2)
ax.plot(centers, f_tau, "k:", lw=2)
ax.legend(fontsize=14)
ax.set_title("Drift")
ax.set_xlabel("$x$", fontsize=24)
ax.set_ylabel("$f(x)$", fontsize=24)

ax2.plot(
    centers,
    a_KM,
    ls="",
    marker=".",
    markersize=8,
    c="k",
    label=r"K-M  ($\tau = 0.5$)",
)
ax2.plot(centers, a_vals, "k", lw=2, label=r"Model ($\tau = 0$)")
ax2.plot(centers, a_tau, "k:", lw=2, label=r"Model ($\tau = 0.5$)")
ax2.legend(fontsize=14)
ax2.set_title("Diffusion")
ax2.set_xlabel("$x$", fontsize=24)
ax2.set_ylabel("$a(x)$", fontsize=24)

fig.tight_layout()
fig.savefig(folder_path / "final_graph.png")